In [1]:
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
with open('../data/names.txt', 'r') as f:
    names = f.read().splitlines()
print(f'Number of names: {len(names)}')

Number of names: 32033


In [3]:
# create vocab
vocab = ['.'] + sorted(list(set(''.join(names))))
stoi = { ch: index for index, ch in enumerate(vocab) }
itos = { index: ch for ch, index in stoi.items() }

In [4]:
# create training set
def build_dataset(words, block_size=3):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for w in word + '.':
            ix = stoi[w]
            Y.append(ix)
            X.append(context)
            context = context[1:] + [ix]
    XS = torch.tensor(X)
    YS = torch.tensor(Y)
    print(XS.shape, YS.shape)
    return XS, YS

import random

random.seed(42)
random.shuffle(names)

pdev, pval, ptest = 0.8, 0.1, 0.1
data_size = len(names)
block_size = 3

sdev = int(data_size * pdev)
sval = int(data_size * (pdev + pval))
Xtr, ytr = build_dataset(names[:sdev], block_size)
Xval, yval = build_dataset(names[sdev: sval], block_size)
Xtest, ytest = build_dataset(names[sval:], block_size)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
# contants
vocab_size = len(vocab)
embed_dim = 10
batch_size = 32
hidden_nuerons = 64

rng = torch.Generator().manual_seed(2147483647)

C = torch.randn(size=(vocab_size, embed_dim), generator=rng)

#layer 1
W1 = torch.randn(size=(embed_dim*block_size, hidden_nuerons), generator=rng) * 5 / 3 * (embed_dim*block_size)**-0.5
b1 = torch.randn(size=(hidden_nuerons, ), generator=rng) * 0.1

#layer 2
W2 = torch.randn(size=(hidden_nuerons, vocab_size), generator=rng) * 0.1
b2 = torch.randn(size=(vocab_size, ), generator=rng) * 0.1

# for batch normalization scale and shift
bngain = torch.randn(size=(1, hidden_nuerons), generator=rng) * 0.1 + 1.0
bnbias = torch.randn(size=(1, hidden_nuerons), generator=rng) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

print(f'Number of parameters: {sum(p.nelement() for p in parameters)}')

for p in parameters:
    p.requires_grad = True


Number of parameters: 4137


In [7]:
learning_rate = 0.1
batch_size = 32

# batching
batch = torch.randint(0, len(Xtr), size=(batch_size, ), generator=rng)
Xb, yb = Xtr[batch], ytr[batch]

#fwd pass
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# linear layer 1
hprebn = embcat @ W1 + b1

bnmeani = 1 / batch_size * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1 / (batch_size - 1) * bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

#activation
h = torch.tanh(hpreact)

# linear layer 2
logits = h @ W2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[torch.arange(batch_size), yb].mean()

# # backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()

loss.backward()
loss


tensor(3.5571, grad_fn=<NegBackward0>)

In [27]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[torch.arange(batch_size), yb] = -1.0 / batch_size
cmp('logprobs', dlogprobs, logprobs)

dprobs = 1.0 / probs * dlogprobs
cmp('probs', dprobs, probs)

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts1 = counts_sum_inv * dprobs
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum = - (counts_sum) ** -2 * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

dcounts2 = torch.ones_like(counts) * dcounts_sum
dcounts = dcounts1 + dcounts2
cmp('counts', dcounts, counts)

dnorm_logits = (counts.clone()) * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

dlogits1 = dnorm_logits.clone()
dlogit_maxes = - (dnorm_logits.sum(1, keepdim=True))
cmp('logit_maxes', dlogit_maxes, logit_maxes)

# dlogit2 = torch.zeros_like(logits)
# dlogit2[torch.arange(batch_size), logits.max(1).indices] = 1.0 
# dlogit2 = dlogit2 * dlogit_maxes

dlogits2 = F.one_hot(logits.max(1).indices, num_classes = logits.shape[1]).float() * dlogit_maxes
dlogits = dlogits1 + dlogits2
cmp('logits', dlogits, logits)

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

# hpreact = bngain * bnraw + bnbias
dbngain = (bnraw * dhpreact).sum(0, keepdim = True)
cmp('bngain', dbngain, bngain)

dbnraw = bngain * dhpreact
cmp('bnraw', dbnraw, bnraw)

dbnbias = dhpreact.sum(0, keepdim=True)
cmp('bnbias', dbnbias, bnbias)

dbndiff1 = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv 
cmp('bnvar', dbnvar, bnvar)

dbndiff2 = 1 / (batch_size - 1) * torch.ones_like(bndiff2) * dbnvar
cmp('bndiff2', dbndiff2, bndiff2)

# bndiff2 = bndiff ** 2
dbndiff2 = (2 * bndiff) * dbndiff2
dbndiff = dbndiff1 + dbndiff2
cmp('bndiff', dbndiff, bndiff)

dhprebn1 = dbndiff.clone()
dhnmeani = (-1.0 * dbndiff).sum(0, keepdim=True)
cmp('bnmeani', dhnmeani, bnmeani)

dhprebn2 = (1.0 / batch_size) * torch.ones_like(hprebn) * dhnmeani
dhprebn = dhprebn2 + dhprebn1
cmp('hprebn', dhprebn, hprebn)

dembcat = dhprebn @ W1.T
dW1 = embcat.T @dhprebn
db1 = dhprebn.sum(0)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

demb = dembcat.view(emb.shape)
cmp('emb', demb, emb)

dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]
cmp('C', dC, C)


logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09
bnraw           | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bnbias 